# TTS Quality Comparison: Qwen3-TTS vs Chatterbox-Turbo

This notebook tests two production-grade open-source TTS models on Colab GPU (T4).

**NO pyttsx3 fallback. Real inference only.**

Output: Two playable WAV files with metadata and quality comparison.

In [ ]:
# Cell 1: Setup and Environment Check
import os
import sys
from pathlib import Path

# Clone/pull repository
if not Path('/content/openmontage-colab').exists():
    print('Cloning openmontage-colab...')
    os.system('git clone --depth 1 https://github.com/jvvghj123-sudo/openmontage-colab /content/openmontage-colab')
else:
    print('Repository already present, pulling latest...')
    os.system('cd /content/openmontage-colab && git pull')

os.chdir('/content/openmontage-colab')
print(f'Working directory: {os.getcwd()}')

# Add to path
if '/content/openmontage-colab' not in sys.path:
    sys.path.insert(0, '/content/openmontage-colab')

print('\nSetup complete.')

In [ ]:
# Cell 2: CUDA and GPU Check
import torch
import subprocess

print('='*70)
print('CUDA / GPU CHECK')
print('='*70)

cuda_available = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if cuda_available else 'CPU'
device_count = torch.cuda.device_count()

print(f'\nTorch version: {torch.__version__}')
print(f'CUDA available: {cuda_available}')
print(f'Device: {device_name}')
print(f'Device count: {device_count}')

if cuda_available:
    print(f'\nGPU Memory:')
    print(f'  Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    print(f'  Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.1f} GB')
    print(f'  Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.1f} GB')
else:
    print('\n⚠️ WARNING: CUDA not available. Tests will run on CPU (slow).')

# Check ffmpeg
ffmpeg_check = subprocess.run(['which', 'ffmpeg'], capture_output=True, text=True)
ffprobe_check = subprocess.run(['which', 'ffprobe'], capture_output=True, text=True)
print(f'\nFFmpeg: {"✓ present" if ffmpeg_check.returncode == 0 else "✗ missing"}')
print(f'FFprobe: {"✓ present" if ffprobe_check.returncode == 0 else "✗ missing"}')

print('\n' + '='*70)

In [ ]:
# Cell 3: Test Qwen3-TTS-12Hz-0.6B-CustomVoice
import time
import json
import subprocess
from pathlib import Path
from IPython.display import Audio, display

print('\n' + '='*70)
print('TEST 1: Qwen3-TTS-12Hz-0.6B-CustomVoice')
print('='*70)

NARRATION = '''A hundred years ago, humanity looked toward the stars and wondered whether we were alone. Tonight, something answered. The signal came from a world no telescope had ever seen before. And buried inside that transmission was a message meant for us.'''

CINEMATIC_INSTRUCTION = "Calm, cinematic narration. Natural pacing. Slight sense of mystery and anticipation. Clear pronunciation. Do not sound like an advertisement."

output_dir = Path('projects/colab-tts-quality')
output_dir.mkdir(parents=True, exist_ok=True)

qwen_result = {
    'model': 'Qwen3-TTS-12Hz-0.6B-CustomVoice',
    'device': None,
    'sample_rate': None,
    'duration': None,
    'file_size': None,
    'generation_time': None,
    'gpu_memory_used': None,
    'status': 'FAIL',
    'error': None,
    'output_path': str(output_dir / 'qwen3_tts_output.wav')
}

try:
    # Record GPU memory before
    gpu_mem_before = torch.cuda.memory_allocated() if cuda_available else 0
    qwen_result['device'] = f"CUDA: {cuda_available}, Device: {device_name}"
    
    # Install Qwen3-TTS
    print('\nInstalling Qwen3-TTS from GitHub...')
    os.system('pip install -q git+https://github.com/QwenLM/Qwen3-TTS.git 2>&1 | tail -5')
    
    # Import and initialize
    print('Initializing Qwen3-TTS model...')
    from qwen3tts.utils import TTS as Qwen3TTS
    
    device = 'cuda' if cuda_available else 'cpu'
    start_time = time.time()
    
    tts = Qwen3TTS('Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice', device=device)
    
    # Generate
    print(f'\nGenerating audio ({len(NARRATION)} characters)...')
    print(f'Instruction: {CINEMATIC_INSTRUCTION[:60]}...')
    
    audio_data = tts.synthesize(
        text=NARRATION,
        instruction=CINEMATIC_INSTRUCTION,
        sampling_rate=12000
    )
    
    gen_time = time.time() - start_time
    qwen_result['generation_time'] = gen_time
    
    # Save
    output_path = output_dir / 'qwen3_tts_output.wav'
    tts.save_wav(audio_data, str(output_path), sampling_rate=12000)
    print(f'Saved to: {output_path}')
    
    # Validate with ffprobe
    result = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'a:0',
         '-show_entries', 'stream=sample_rate,channels,duration',
         '-show_entries', 'format=size', '-of', 'json', str(output_path)],
        capture_output=True, text=True, timeout=10
    )
    
    if result.returncode == 0:
        data = json.loads(result.stdout)
        if data.get('streams'):
            stream = data['streams'][0]
            fmt = data.get('format', {})
            qwen_result['sample_rate'] = int(stream.get('sample_rate', 0))
            qwen_result['duration'] = float(stream.get('duration', 0))
            qwen_result['file_size'] = int(fmt.get('size', 0))
            qwen_result['status'] = 'PASS'
            print(f'\n✓ FFmpeg validation PASSED')
            print(f'  Sample rate: {qwen_result["sample_rate"]} Hz')
            print(f'  Duration: {qwen_result["duration"]:.2f} seconds')
            print(f'  File size: {qwen_result["file_size"] / 1024:.1f} KB')
        else:
            raise RuntimeError('No audio stream in output')
    else:
        raise RuntimeError(f'ffprobe failed: {result.stderr}')
    
    # GPU memory
    if cuda_available:
        gpu_mem_after = torch.cuda.memory_allocated()
        qwen_result['gpu_memory_used'] = (gpu_mem_after - gpu_mem_before) / 1024 / 1024
        print(f'  GPU memory used: {qwen_result["gpu_memory_used"]:.1f} MB')
    
    print(f'  Generation time: {gen_time:.2f} seconds')
    print(f'\n✓ Qwen3-TTS REAL TEST: PASS')
    
except Exception as e:
    qwen_result['error'] = str(e)
    qwen_result['status'] = 'FAIL'
    print(f'\n✗ Qwen3-TTS REAL TEST: FAIL')
    print(f'Error: {e}')
    import traceback
    traceback.print_exc()

print('\n' + '='*70)

In [ ]:
# Cell 4: Play Qwen3-TTS Audio and Metadata
if qwen_result['status'] == 'PASS':
    print('\nQWEN3-TTS OUTPUT')
    print('='*70)
    print(f'Model: {qwen_result["model"]}')
    print(f'Status: {qwen_result["status"]}')
    print(f'Device: {qwen_result["device"]}')
    print(f'Sample rate: {qwen_result["sample_rate"]} Hz')
    print(f'Duration: {qwen_result["duration"]:.2f} seconds')
    print(f'File size: {qwen_result["file_size"] / 1024:.1f} KB')
    print(f'Generation time: {qwen_result["generation_time"]:.2f} seconds')
    if qwen_result.get('gpu_memory_used'):
        print(f'GPU memory used: {qwen_result["gpu_memory_used"]:.1f} MB')
    print('\nAudio playback:')
    display(Audio(qwen_result['output_path']))
else:
    print(f'\n✗ Qwen3-TTS test failed: {qwen_result["error"]}')

print('='*70)

In [ ]:
# Cell 5: Test Chatterbox-Turbo
print('\n' + '='*70)
print('TEST 2: Chatterbox-Turbo')
print('='*70)

chatterbox_result = {
    'model': 'Chatterbox-Turbo',
    'device': None,
    'sample_rate': None,
    'duration': None,
    'file_size': None,
    'generation_time': None,
    'gpu_memory_used': None,
    'status': 'FAIL',
    'error': None,
    'output_path': str(output_dir / 'chatterbox_output.wav')
}

try:
    # Record GPU memory before
    gpu_mem_before = torch.cuda.memory_allocated() if cuda_available else 0
    chatterbox_result['device'] = f"CUDA: {cuda_available}, Device: {device_name}"
    
    # Install Chatterbox
    print('\nInstalling Chatterbox from GitHub...')
    os.system('pip install -q git+https://github.com/resemble-ai/chatterbox.git 2>&1 | tail -5')
    
    # Also ensure soundfile is installed
    print('Installing soundfile dependency...')
    os.system('pip install -q soundfile')
    
    # Import and initialize
    print('Initializing Chatterbox-Turbo model...')
    from chatterbox import ChatterboxTurbo
    
    device = 'cuda' if cuda_available else 'cpu'
    start_time = time.time()
    
    tts = ChatterboxTurbo(model_id='ResembleAI/chatterbox-turbo', device=device)
    
    # Generate
    print(f'\nGenerating audio ({len(NARRATION)} characters)...')
    
    audio_data, sample_rate = tts.synthesize(
        text=NARRATION,
        voice='default'
    )
    
    gen_time = time.time() - start_time
    chatterbox_result['generation_time'] = gen_time
    chatterbox_result['sample_rate'] = sample_rate
    
    # Save
    import soundfile as sf
    output_path = output_dir / 'chatterbox_output.wav'
    sf.write(str(output_path), audio_data, sample_rate)
    print(f'Saved to: {output_path}')
    
    # Validate with ffprobe
    result = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'a:0',
         '-show_entries', 'stream=sample_rate,channels,duration',
         '-show_entries', 'format=size', '-of', 'json', str(output_path)],
        capture_output=True, text=True, timeout=10
    )
    
    if result.returncode == 0:
        data = json.loads(result.stdout)
        if data.get('streams'):
            stream = data['streams'][0]
            fmt = data.get('format', {})
            chatterbox_result['sample_rate'] = int(stream.get('sample_rate', 0))
            chatterbox_result['duration'] = float(stream.get('duration', 0))
            chatterbox_result['file_size'] = int(fmt.get('size', 0))
            chatterbox_result['status'] = 'PASS'
            print(f'\n✓ FFmpeg validation PASSED')
            print(f'  Sample rate: {chatterbox_result["sample_rate"]} Hz')
            print(f'  Duration: {chatterbox_result["duration"]:.2f} seconds')
            print(f'  File size: {chatterbox_result["file_size"] / 1024:.1f} KB')
        else:
            raise RuntimeError('No audio stream in output')
    else:
        raise RuntimeError(f'ffprobe failed: {result.stderr}')
    
    # GPU memory
    if cuda_available:
        gpu_mem_after = torch.cuda.memory_allocated()
        chatterbox_result['gpu_memory_used'] = (gpu_mem_after - gpu_mem_before) / 1024 / 1024
        print(f'  GPU memory used: {chatterbox_result["gpu_memory_used"]:.1f} MB')
    
    print(f'  Generation time: {gen_time:.2f} seconds')
    print(f'\n✓ Chatterbox-Turbo REAL TEST: PASS')
    
except Exception as e:
    chatterbox_result['error'] = str(e)
    chatterbox_result['status'] = 'FAIL'
    print(f'\n✗ Chatterbox-Turbo REAL TEST: FAIL')
    print(f'Error: {e}')
    import traceback
    traceback.print_exc()

print('\n' + '='*70)

In [ ]:
# Cell 6: Play Chatterbox Audio and Metadata
if chatterbox_result['status'] == 'PASS':
    print('\nCHATTERBOX-TURBO OUTPUT')
    print('='*70)
    print(f'Model: {chatterbox_result["model"]}')
    print(f'Status: {chatterbox_result["status"]}')
    print(f'Device: {chatterbox_result["device"]}')
    print(f'Sample rate: {chatterbox_result["sample_rate"]} Hz')
    print(f'Duration: {chatterbox_result["duration"]:.2f} seconds')
    print(f'File size: {chatterbox_result["file_size"] / 1024:.1f} KB')
    print(f'Generation time: {chatterbox_result["generation_time"]:.2f} seconds')
    if chatterbox_result.get('gpu_memory_used'):
        print(f'GPU memory used: {chatterbox_result["gpu_memory_used"]:.1f} MB')
    print('\nAudio playback:')
    display(Audio(chatterbox_result['output_path']))
else:
    print(f'\n✗ Chatterbox-Turbo test failed: {chatterbox_result["error"]}')

print('='*70)

In [ ]:
# Cell 7: Comparison Table and Quality Checklist
print('\n' + '='*70)
print('TTS QUALITY COMPARISON')
print('='*70)

# Print comparison table
print('\n' + '-'*130)
print(f"{'Model':<25} {'Real GPU':<12} {'Duration':<12} {'Sample Rate':<15} {'Gen Time':<12} {'File Size':<15} {'GPU Mem':<12} {'Status':<8}")
print('-'*130)

for name, res in [('Qwen3-TTS', qwen_result), ('Chatterbox-Turbo', chatterbox_result)]:
    gpu = 'Yes' if 'CUDA: True' in (res.get('device') or '') else 'No'
    duration = f"{res['duration']:.1f}s" if res.get('duration') else '—'
    sr = f"{res['sample_rate']} Hz" if res.get('sample_rate') else '—'
    gen_time = f"{res['generation_time']:.2f}s" if res.get('generation_time') else '—'
    file_size = f"{res['file_size'] / 1024:.1f} KB" if res.get('file_size') else '—'
    gpu_mem = f"{res['gpu_memory_used']:.0f} MB" if res.get('gpu_memory_used') else '—'
    status = res['status']
    
    print(f"{name:<25} {gpu:<12} {duration:<12} {sr:<15} {gen_time:<12} {file_size:<15} {gpu_mem:<12} {status:<8}")

print('-'*130)

# Save report
report = {
    'timestamp': str(time.strftime('%Y-%m-%d %H:%M:%S')),
    'narration_length': len(NARRATION),
    'device': device_name,
    'cuda_available': cuda_available,
    'qwen3': qwen_result,
    'chatterbox': chatterbox_result
}

report_path = output_dir / 'tts_quality_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'\nReport saved to: {report_path}')

print('\n' + '='*70)
print('LISTENING CHECKLIST (evaluate manually in Colab audio players above)')
print('='*70)

checklist = """
For each model, evaluate:

  □ Natural voice? (not robotic/synthetic-sounding)
  □ Robotic artifacts? (clicks, glitches, unnaturalness)
  □ Pronunciation clarity? (correct word pronunciation)
  □ Pacing and pauses? (natural rhythm, appropriate silence)
  □ Emotional expression? (matches the cinematic tone requested)
  □ Background noise? (hum, hiss, static)
  □ Clipping or distortion? (audio cutoff, harshness)
  □ Suitable for YouTube narration? (professional quality)
  □ Suitable for long-form content? (won't tire listener)

Recommendation (circle one):
  - QWEN3-TTS is better
  - CHATTERBOX-TURBO is better
  - Both acceptable, choose by speed/vram
  - Both unacceptable, need different model
"""

print(checklist)

print('='*70)
print('TEST COMPLETE')
print('='*70)

## Summary

**Qwen3-TTS:** Generated `{qwen_result['output_path']}` with {qwen_result['sample_rate']} Hz, {qwen_result['duration']:.1f}s, status: {qwen_result['status']}

**Chatterbox-Turbo:** Generated `{chatterbox_result['output_path']}` with {chatterbox_result['sample_rate']} Hz, {chatterbox_result['duration']:.1f}s, status: {chatterbox_result['status']}

**Next:** Listen to both audio outputs above using the IPython audio players. Complete the quality checklist and choose the model for production Stage 2 TTS.

Do NOT integrate into the pipeline until manual evaluation is complete.